In [0]:
cloudsrc="/Volumes/catalog1_we47/schema_we47/clouddatalake/sourcesystemdata/"#s3 
bronzetgt="/Volumes/catalog1_we47/schema_we47/bronze/ourtargetlocation/"

#Databricks workspace, such as an S3 bucket or a Unity Catalog volume.
ckptlocation="/Volumes/catalog1_we47/schema_we47/clouddatalake/ckpt/_checkpoint"#stores the files copied information post write is successful
schemalocation="/Volumes/catalog1_we47/schema_we47/clouddatalake/_schema"#stores the inferred schema of the source data
df1=spark.readStream.format("cloudFiles")\
.option("cloudFiles.format","csv")\
.option("cloudFiles.maxFilesPerTrigger",1)\
.option("cloudFiles.inferColumnTypes",True)\
.option("cloudFiles.schemaEvolutionMode","addNewColumns")\
.option("checkpointLocation", ckptlocation)\
.option("cloudFiles.schemaLocation", schemalocation)\
.option("header",True)\
.load(cloudsrc)

In [0]:
#realtime trigger is not possible in free serverless
#writeStream will read data from df1 (materialized here) and write to bronzetgt using the schema generated by reader and checkpoint info stored
df1.writeStream.trigger(availableNow=True)\
.option("checkpointLocation", ckptlocation)\
.option("cloudFiles.schemaLocation", schemalocation)\
.option("mergeSchema", "true") \
.outputMode("append")\
.start(bronzetgt)
#.option("mergeSchema", "true") \